# Session 2: from soil strength to footing size
**A. Real-world problem.** A 600 kN column must stand on soil whose allowable bearing
pressure is 150 kPa. How wide must the square footing be once we count the footing's own
weight and a factor of safety? And where does the allowable pressure come from?
Educational model only: never use it to design a real foundation.

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
GAMMA_CONCRETE = 24.0     # unit weight of reinforced concrete, kN/m^3 (typical textbook value)

## Step 1: contact pressure with the footing's own weight
Soil carries the column load P plus the footing weight W = 24 * B^2 * t (t = thickness).

    q = (P + 24 B^2 t) / B^2 = P / B^2 + 24 t

The allowable pressure q_allow already contains a factor of safety, so we need q <= q_allow:

    B >= sqrt( P / (q_allow - 24 t) )

In [ ]:
def contact_pressure(load_kn, side_m, thickness_m=0.6):
    """Pressure (kPa) on the soil under a square footing including its own weight."""
    return (load_kn + GAMMA_CONCRETE * side_m ** 2 * thickness_m) / side_m ** 2


def required_side(load_kn, q_allow_kpa, thickness_m=0.6, step=0.05):
    """Smallest square side (m), rounded UP to `step`, with contact pressure <= q_allow."""
    net = q_allow_kpa - GAMMA_CONCRETE * thickness_m
    if net <= 0:
        raise ValueError("the footing weight alone uses up the allowable pressure")
    exact = math.sqrt(load_kn / net)
    return math.ceil(exact / step - 1e-9) * step

## Step 2: where allowable pressure comes from (Terzaghi, strip footing)
Terzaghi's bearing capacity equation for a long strip footing:

    q_ult = c N_c + q N_q + 0.5 gamma B N_gamma

c = cohesion (kPa), q = soil pressure beside the footing at its base = gamma * D_f (kPa),
gamma = soil unit weight (kN/m^3), B = width (m). For saturated clay in short-term loading
(phi = 0) the factors are N_c = 5.7, N_q = 1, N_gamma = 0, so q_ult = 5.7 c + gamma D_f.
The allowable pressure is q_allow = q_ult / FS, with a factor of safety FS of about 3.
For phi > 0, N_q and N_c follow from formulas below (N_gamma needs a table: not computed here).

In [ ]:
def terzaghi_nq(phi_deg):
    p = math.radians(phi_deg)
    a = math.exp((0.75 * math.pi - p / 2) * math.tan(p))
    return a * a / (2 * math.cos(math.pi / 4 + p / 2) ** 2)


def terzaghi_nc(phi_deg):
    """N_c = (N_q - 1) cot(phi); for phi = 0 the standard Terzaghi value 5.7 is returned."""
    if phi_deg == 0:
        return 5.7
    return (terzaghi_nq(phi_deg) - 1) / math.tan(math.radians(phi_deg))


def q_ult_strip_phi0(cohesion_kpa, gamma_knm3=0.0, depth_m=0.0):
    """Ultimate bearing pressure (kPa) of a strip footing on clay with phi = 0."""
    return 5.7 * cohesion_kpa + gamma_knm3 * depth_m


def allowable_from_ultimate(q_ult_kpa, fs=3.0):
    return q_ult_kpa / fs


def factor_of_safety(q_ult_kpa, q_applied_kpa):
    return q_ult_kpa / q_applied_kpa

## Step 3: eccentric loads and the middle third
If the load acts a distance e from the footing centre, one edge is pressed harder.
For a footing of width B (length L), the edge pressures are

    q_max,min = P / (B L) * (1 +- 6 e / B)

The small edge pressure reaches zero when e = B/6 (the "middle third" rule). Beyond that
the soil cannot pull on the footing: part of the base lifts off and the contact length
shrinks to 3 (B/2 - e), giving q_max = 2 P / (3 L (B/2 - e)).

In [ ]:
def edge_pressures(load_kn, width_m, e_m, length_m=1.0):
    """Return (q_max, q_min, full_contact). Use length_m = 1 for a load per metre of a strip."""
    if e_m <= width_m / 6 * (1 + 1e-12):
        mean = load_kn / (width_m * length_m)
        k = 6 * e_m / width_m
        return mean * (1 + k), mean * (1 - k), True
    contact = 3 * (width_m / 2 - e_m)
    if contact <= 0:
        raise ValueError("load acts outside the footing")
    return 2 * load_kn / (contact * length_m), 0.0, False

## Step 4: see it

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    B = required_side(600, 150)
    print("600 kN on 150 kPa, t = 0.6 m: exact", round(math.sqrt(600 / (150 - 14.4)), 3), "-> B =", round(B, 2), "m")
    print("contact pressure at that size:", round(contact_pressure(600, B), 1), "kPa; at 2.10 m:", round(contact_pressure(600, 2.10), 1), "kPa")
    print("clay c = 50 kPa strip: q_ult =", q_ult_strip_phi0(50), "kPa, q_allow (FS 3) =", round(allowable_from_ultimate(285), 1), "kPa")
    print("phi = 30: Nq =", round(terzaghi_nq(30), 1), " Nc =", round(terzaghi_nc(30), 1))
    print("strip 2 m wide, 300 kN/m, e = 0.2:", [round(v, 1) if not isinstance(v, bool) else v for v in edge_pressures(300, 2.0, 0.2)])
    print("e = B/6:", [round(v, 1) if not isinstance(v, bool) else v for v in edge_pressures(300, 2.0, 2.0 / 6)])
    print("e = 0.5:", [round(v, 1) if not isinstance(v, bool) else v for v in edge_pressures(300, 2.0, 0.5)])

    qa = np.linspace(60, 400, 100)
    fig, ax = plt.subplots(figsize=(6, 4))
    for P, col in ((300, "C0"), (600, "C1"), (1200, "C2")):
        ax.plot(qa, [math.sqrt(P / (q - 14.4)) for q in qa], color=col, label=f"P = {P} kN")
    ax.plot([150], [math.sqrt(600 / (150 - 14.4))], "ko")
    ax.set_xlabel("allowable bearing pressure q_allow (kPa)")
    ax.set_ylabel("required square side B (m)")
    ax.set_title("Weaker soil needs a much wider footing (t = 0.6 m)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "required_width.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    x = np.linspace(-1, 1, 201)
    for e, col in ((0.0, "C0"), (0.2, "C1"), (1.0 / 3, "C2"), (0.5, "C3")):
        qmax, qmin, full = edge_pressures(300, 2.0, e)
        if full:
            q = (300 / 2.0) * (1 + 12 * e * x / 2.0 ** 2)     # straight-line pressure, x measured from the centre
        else:
            xl = 1.0 - 3 * (1.0 - e)
            q = np.where(x >= xl, qmax * (x - xl) / (1.0 - xl), 0.0)
        ax2.plot(x, q, color=col, label=f"e = {e:.2f} m")
    ax2.set_xlabel("position across the footing (m)")
    ax2.set_ylabel("pressure on soil (kPa)")
    ax2.set_title("Eccentric load, strip 2 m wide, 300 kN/m")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "eccentric_pressure.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print `required_side(600, 150, t)` for t = 0.3, 0.6 and 1.0 m and explain the trend.
2. **Stretch.** Add a function `strip_width(line_load, cohesion, fs=3)` that returns the strip width rounded up to 0.05 m, and test it on 200 kN/m and c = 50 kPa.
3. **Explain.** Explain in your own words why the eccentric pressure can never be negative in the model and what actually happens to the footing beyond e = B / 6.

## Assessment
- **A. Conceptual.** Why must the eccentricity stay within the middle third (e at most B / 6) for a footing on soil? What happens if it does not?
- **B. Numerical.** A strip footing carries 200 kN/m on clay with c = 50 kPa. Using phi = 0, a factor of safety of 3 and ignoring the footing weight, what width is needed? Give the width you would build in 0.05 m steps.
- **C. Graph and simulation interpretation.** From the required-width graph, what happens to the footing side for a 600 kN column when the allowable pressure falls from 150 kPa to 75 kPa? Compute it.
- **D. Design and optimisation challenge.** Design: the 600 kN column in the worked example also carries a moment that makes the load act 0.2 m off-centre in a square footing (t = 0.6 m, 150 kPa allowable including the footing weight). Find the smallest side in 0.05 m steps with e at most B / 6 and a peak pressure no greater than 150 kPa. How much more plan area does the eccentricity cost?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    exact = math.sqrt(600 / (150 - 24 * 0.6))
    checks.append(("worked example: 600 kN, 150 kPa, t = 0.6 m needs exactly 2.104 m, rounded up to 2.15 m",
                   abs(exact - 2.1038) < 1e-3 and abs(L.required_side(600, 150) - 2.15) < 1e-9))
    checks.append(("worked example: at 2.15 m the contact pressure is 144.2 kPa, at 2.10 m it is 150.5 kPa (fails)",
                   abs(L.contact_pressure(600, 2.15) - 144.2) < 0.05 and L.contact_pressure(600, 2.10) > 150.0 > L.contact_pressure(600, 2.15)))
    checks.append(("hand calculation: contact pressure = P / B^2 + 24 t = 600/4 + 14.4 = 164.4 kPa for B = 2 m",
                   abs(L.contact_pressure(600, 2.0) - 164.4) < 1e-9))
    checks.append(("a thicker footing needs a wider base (t = 1.0 m gives a larger side than t = 0.4 m)",
                   L.required_side(600, 150, 1.0) > L.required_side(600, 150, 0.4)))
    checks.append(("weaker soil needs a wider footing (75 kPa gives more than 150 kPa)", L.required_side(600, 75) > L.required_side(600, 150)))
    try:
        L.required_side(100, 14.0)
        ok = False
    except ValueError:
        ok = True
    checks.append(("edge: allowable pressure below the footing weight is rejected", ok))
    checks.append(("Terzaghi phi = 0 strip: q_ult = 5.7 * 50 = 285 kPa", abs(L.q_ult_strip_phi0(50) - 285.0) < 1e-9))
    checks.append(("with D_f = 1 m and gamma = 18 kN/m^3 the capacity is 285 + 18 = 303 kPa", abs(L.q_ult_strip_phi0(50, 18.0, 1.0) - 303.0) < 1e-9))
    checks.append(("allowable pressure with FS = 3 is 95 kPa", abs(L.allowable_from_ultimate(285.0) - 95.0) < 1e-9))
    checks.append(("a 150 kPa pressure on a 285 kPa capacity gives FS = 1.9", abs(L.factor_of_safety(285.0, 150.0) - 1.9) < 1e-9))
    checks.append(("N_q = 1 at phi = 0", abs(L.terzaghi_nq(0.0) - 1.0) < 1e-9))
    checks.append(("the N_c formula tends to 5.7 as phi tends to 0 (5.71 at 0.0001 degrees)", abs(L.terzaghi_nc(1e-4) - 5.7) < 0.02 and L.terzaghi_nc(0) == 5.7))
    checks.append(("phi = 30 degrees: N_q = 22.5 and N_c = 37.2",
                   abs(L.terzaghi_nq(30) - 22.5) < 0.05 and abs(L.terzaghi_nc(30) - 37.2) < 0.05))
    qmax, qmin, full = L.edge_pressures(300, 2.0, 0.2)
    checks.append(("worked example: strip B = 2 m, 300 kN/m, e = 0.2 m gives 240 and 60 kPa", abs(qmax - 240) < 1e-9 and abs(qmin - 60) < 1e-9 and full))
    qmax, qmin, full = L.edge_pressures(300, 2.0, 2.0 / 6)
    checks.append(("edge: at e = B/6 the small edge pressure is exactly 0 and the large one is twice the mean (300 kPa)",
                   abs(qmin) < 1e-9 and abs(qmax - 300) < 1e-9 and full))
    qmax, qmin, full = L.edge_pressures(300, 2.0, 0.34)
    checks.append(("just beyond B/6 (e = 0.34 m) the base starts to lift off", not full and qmin == 0.0))
    ok = True
    for e in np.linspace(0, 2.0 / 6, 25):
        ok &= L.edge_pressures(300, 2.0, e)[1] >= -1e-9
    checks.append(("scan: no tension for 25 eccentricities from 0 to B/6", ok))
    qmax, qmin, full = L.edge_pressures(300, 2.0, 0.5)
    contact = 3 * (1.0 - 0.5)
    checks.append(("lift-off case e = 0.5 m: contact length 1.5 m, q_max = 400 kPa, triangle carries the 300 kN",
                   abs(qmax - 400) < 1e-9 and abs(0.5 * qmax * contact - 300) < 1e-9 and not full))
    checks.append(("lift-off case: triangle centroid sits 0.5 m from the centre = e (moment balance)", abs((1.0 - contact / 3) - 0.5) < 1e-12))
    checks.append(("design: strip load 200 kN/m on c = 50 kPa clay with FS 3 needs 2.105 m, built 2.15 m",
                   abs(200 / L.allowable_from_ultimate(L.q_ult_strip_phi0(50)) - 2.1053) < 1e-3))
    checks.append(("ignoring the footing weight the side would be exactly 2.0 m (600 / 150 = 4)", abs(math.sqrt(600 / 150) - 2.0) < 1e-12))
    checks.append(("assessment B: strip 200 kN/m on 2.15 m gives 93 kPa, below 95 kPa", abs(200 / 2.15 - 93.02) < 0.01 and 200 / 2.15 < 95.0))
    b75 = L.required_side(600, 75)
    checks.append(("assessment C: at 75 kPa the side is 3.15 m, 46% wider and more than double the area of 2.15 m",
                   abs(b75 - 3.15) < 1e-9 and abs(b75 / 2.15 - 1.465) < 0.001 and b75 ** 2 / 2.15 ** 2 > 2.0))
    def peak(b):
        return L.edge_pressures(600, b, 0.2, length_m=b)[0] + 24 * 0.6
    best = next(b for b in np.arange(2.15, 4.0, 0.05) if 0.2 <= b / 6 and peak(b) <= 150.0)
    checks.append(("assessment D: with e = 0.2 m, B = 2.55 m gives 150.1 kPa (fails), 2.60 m gives 144.1 kPa; the smallest side is 2.60 m",
                   abs(peak(2.55) - 150.1) < 0.05 and abs(peak(2.60) - 144.1) < 0.05 and abs(best - 2.6) < 1e-9))
    checks.append(("assessment D: e = 0.2 m is below B/6 = 0.43 m and the area is 6.76 m2 against 4.62 m2, 46% more",
                   abs(2.6 / 6 - 0.4333) < 1e-3 and abs(2.6 ** 2 - 6.76) < 1e-9 and abs(2.15 ** 2 - 4.6225) < 1e-9 and abs(2.6 ** 2 / 2.15 ** 2 - 1.462) < 0.001))
    checks.append(("at e = 0 the strip pressure is flat 150 kPa", L.edge_pressures(300, 2.0, 0.0)[:2] == (150.0, 150.0)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
